In [3]:
import os
print(os.listdir('/kaggle/input'))


['datasets']


In [4]:
import torch, numpy as np, pandas as pd, os, glob, time
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoProcessor

dev = 'cuda'
OUT = '/kaggle/working'

# path নিজে খুঁজে নেয় — dataset কোথায় mount হলো তা নিয়ে ভাবতে হবে না
TXTP = glob.glob('/kaggle/input/**/texts.parquet', recursive=True)[0]
IMGD = glob.glob('/kaggle/input/**/img384', recursive=True)[0]
print('texts :', TXTP)
print('images:', IMGD)

texts = pd.read_parquet(TXTP)
img_hashes = sorted(h[:-4] for h in os.listdir(IMGD))    # sorted বাধ্যতামূলক — এটাই mapping
print('texts:', len(texts), '| images:', len(img_hashes))   # 14581 | 14598 হওয়ার কথা

def norm(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True).clip(1e-8)

def to_tensor(out):
    if torch.is_tensor(out): return out
    if getattr(out, 'pooler_output', None) is not None: return out.pooler_output
    return out.last_hidden_state[:, 0]

class ImgDS(Dataset):
    def __init__(self, hashes, proc): self.h, self.p = hashes, proc
    def __len__(self): return len(self.h)
    def __getitem__(self, i):
        im = Image.open(f'{IMGD}/{self.h[i]}.jpg').convert('RGB')
        return self.p(images=im, return_tensors='pt')['pixel_values'][0]

@torch.no_grad()
def embed_images(proc, fn, bs=32):
    dl = DataLoader(ImgDS(img_hashes, proc), batch_size=bs, num_workers=4, pin_memory=True)
    out, t0 = [], time.time()
    for i, b in enumerate(dl):
        out.append(to_tensor(fn(b.to(dev, torch.float16))).float().cpu().numpy())
        if i % 50 == 0:
            print(f'  {i*bs}/{len(img_hashes)}  {time.time()-t0:.0f}s', flush=True)
    return norm(np.concatenate(out))

@torch.no_grad()
def embed_texts(fn, bs=128):
    out, t0 = [], time.time()
    for i in range(0, len(texts), bs):
        out.append(fn(texts.text.iloc[i:i+bs].tolist()))
        if (i // bs) % 20 == 0:
            print(f'  {i}/{len(texts)}  {time.time()-t0:.0f}s', flush=True)
    return norm(np.concatenate(out))


texts : /kaggle/input/datasets/gpttry/astro1/essentials/texts.parquet
images: /kaggle/input/datasets/gpttry/astro1/img384
texts: 14581 | images: 14598


In [5]:
M = 'google/siglip-so400m-patch14-384'      # cache ঠিক 384px-এ, তাই native resolution
sig   = AutoModel.from_pretrained(M, torch_dtype=torch.float16).to(dev).eval()
sproc = AutoProcessor.from_pretrained(M)

si = embed_images(sproc.image_processor, sig.get_image_features)
np.save(f'{OUT}/emb_siglip_img.npy', si); print('siglip img', si.shape)

def sig_txt(batch):
    tk = sproc.tokenizer(batch, padding='max_length', truncation=True,
                         max_length=64, return_tensors='pt').to(dev)   # SigLIP-এ max_length বাধ্যতামূলক
    return to_tensor(sig.get_text_features(**tk)).float().cpu().numpy()

st = embed_texts(sig_txt)
np.save(f'{OUT}/emb_siglip_txt.npy', st); print('siglip txt', st.shape)


config.json:   0%|          | 0.00/576 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.51G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  0/14598  9s
  1600/14598  81s
  3200/14598  165s
  4800/14598  249s
  6400/14598  333s
  8000/14598  417s
  9600/14598  501s
  11200/14598  585s
  12800/14598  669s
  14400/14598  753s
siglip img (14598, 1152)
  0/14581  1s
  2560/14581  11s
  5120/14581  22s
  7680/14581  33s
  10240/14581  43s
  12800/14581  54s
siglip txt (14581, 1152)


In [7]:
import torch, numpy as np, pandas as pd, os, glob, time
pd.DataFrame({'hash': img_hashes}).to_parquet(f'{OUT}/img_hashes_siglip.parquet')
texts[['hash']].to_parquet(f'{OUT}/txt_hashes_siglip.parquet')
print('saved:', os.listdir(OUT))


saved: ['.virtual_documents', 'img_hashes_siglip.parquet', 'emb_siglip_txt.npy', 'txt_hashes_siglip.parquet', 'emb_siglip_img.npy']


In [ ]:
# ===== NOTEBOOK B — CELL 4 : GATE 4, SigLIP sanity =====
# feature বানানোর আগে দেখে নিই embedding আদৌ label আলাদা করছে কিনা।
# index mapping ভুল হলে এখানেই ধরা পড়বে, LightGBM চালিয়ে সময় নষ্ট হবে না।
import numpy as np, pandas as pd, glob, os

ESS = os.path.dirname(glob.glob('/kaggle/input/**/meta_train.parquet', recursive=True)[0])
OUT = '/kaggle/working'
LABELS = ['same_figure','same_paper','related_papers','unrelated_papers']

mtr = pd.read_parquet(f'{ESS}/meta_train.parquet')
mte = pd.read_parquet(f'{ESS}/meta_test.parquet')
for d in (mtr, mte):
    d['combo'] = np.where(d.t1 < d.t2, d.t1+'+'+d.t2, d.t2+'+'+d.t1)
mtr['y'] = mtr.label.astype(str)

# দুই encoder-এর hash→row map। SigLIP-এরটা B-তে বানানো, CLIP-এরটা essentials থেকে।
iix = {h: i for i, h in enumerate(pd.read_parquet(f'{OUT}/img_hashes_siglip.parquet').hash.values)}
tix = {h: i for i, h in enumerate(pd.read_parquet(f'{OUT}/txt_hashes_siglip.parquet').hash.values)}
iix_c = {h: i for i, h in enumerate(pd.read_parquet(f'{ESS}/img_hashes.parquet').hash.values)}
tix_c = {h: i for i, h in enumerate(pd.read_parquet(f'{ESS}/txt_hashes.parquet').hash.values)}

def center(x):
    z = x - x.mean(0, keepdims=True)
    return z / np.linalg.norm(z, axis=1, keepdims=True).clip(1e-8)

E = {
  'sig':  (np.load(f'{OUT}/emb_siglip_img.npy'), np.load(f'{OUT}/emb_siglip_txt.npy'), iix, tix),
  'clip': (np.load(f'{ESS}/emb_clip_img.npy'),   np.load(f'{ESS}/emb_clip_txt.npy'),   iix_c, tix_c),
  'sci':  (None, center(np.load(f'{ESS}/emb_sci_txt.npy')),                            None,  tix_c),
  'dino': (center(np.load(f'{ESS}/emb_dino_img.npy')), None,                           iix_c, None),
}

def side(hashes, types, space):
    """এক পাশের embedding matrix। dim আর hardcode নয় — encoder থেকেই নেওয়া।"""
    ie, te, ii, ti = E[space]
    dim = (ie if ie is not None else te).shape[1]
    out = np.zeros((len(hashes), dim), dtype=np.float32)
    for i, (h, t) in enumerate(zip(hashes, types)):
        if t == 'image' and ie is not None: out[i] = ie[ii[h]]
        elif t == 'text' and te is not None: out[i] = te[ti[h]]
    return out

for sp in ['sig', 'clip']:
    c = (side(mtr.h1.values, mtr.t1.values, sp) * side(mtr.h2.values, mtr.t2.values, sp)).sum(1)
    t = mtr.assign(cos=c).pivot_table(index='combo', columns='label',
                                      values='cos', aggfunc='mean', observed=True)
    print(f'\n[{sp}] mean cosine'); print(t.round(3))


In [ ]:
# ===== NOTEBOOK B — CELL 5 : SigLIP বনাম CLIP, একই LightGBM =====
# Plan A-র রেফারেন্স (CLIP+domain): image+text 0.3964 | text+text 0.5529 | image+image 0.4548
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

def pair_block(e1, e2, tag):
    cos = (e1*e2).sum(1); l2 = np.linalg.norm(e1-e2, axis=1)
    X = np.hstack([np.abs(e1-e2), e1*e2, cos[:,None], l2[:,None]])
    n = e1.shape[1]
    return X, [f'{tag}_d{i}' for i in range(n)] + [f'{tag}_p{i}' for i in range(n)] + [f'{tag}_cos', f'{tag}_l2']

def build(m, combo, spaces):
    d = m[m.combo == combo].reset_index(drop=True)
    blocks, cols = [], []
    for sp in spaces:
        X, c = pair_block(side(d.h1.values, d.t1.values, sp),
                          side(d.h2.values, d.t2.values, sp), sp)
        blocks.append(X); cols += c
    mn = np.minimum(d.len1, d.len2).values; mx = np.maximum(d.len1, d.len2).values
    blocks.append(np.c_[mn, mx, mx/np.maximum(mn,1)]); cols += ['len_min','len_max','len_ratio']
    return d, np.hstack(blocks).astype(np.float32), cols

def oof_score(combo, spaces):
    d, X, _ = build(mtr, combo, spaces)
    classes = sorted(d.y.unique())
    y = d.y.map({c:i for i,c in enumerate(classes)}).values
    oof = np.zeros((len(y), len(classes)))
    for tr_i, va_i in StratifiedKFold(5, shuffle=True, random_state=0).split(X, y):
        clf = lgb.LGBMClassifier(objective='multiclass', num_class=len(classes),
                                 n_estimators=600, learning_rate=0.05, num_leaves=31,
                                 colsample_bytree=0.3, subsample=0.8, subsample_freq=1,
                                 class_weight='balanced', verbose=-1, n_jobs=-1)
        clf.fit(np.vstack([X[tr_i]]*2), np.concatenate([y[tr_i]]*2))   # symmetry swap
        oof[va_i] = clf.predict_proba(X[va_i])
    s = f1_score(d.y, [classes[i] for i in oof.argmax(1)], average='macro')
    print(f'  {"+".join(spaces):18s} dim={X.shape[1]:5d}  macro-F1 {s:.4f}', flush=True)
    return s, d, oof, classes

# প্রতি subset-এ তিনটা করে: শুধু CLIP (A-র baseline), শুধু SigLIP, দুটো একসাথে
ARMS = {
 'image+text':  [['clip'], ['sig'], ['clip','sig']],
 'text+text':   [['clip','sci'], ['sig'], ['clip','sci','sig']],
 'image+image': [['clip','dino'], ['sig'], ['clip','dino','sig']],
}
store = {}
for combo, arms in ARMS.items():
    print(combo)
    for sp in arms:
        store[(combo, tuple(sp))] = oof_score(combo, sp)


In [ ]:
# ===== NOTEBOOK B — CELL 8 : স্বয়ংসম্পূর্ণ → সরাসরি submission.csv =====
# আগের কোনো cell চালানো লাগবে না। kernel refresh হলেও শুধু এই cell চালালেই হবে।
# pickle নেই — যা লাগে সব CSV আর ছোট .npy-তে।
import os, glob, time, json
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

OUT, LABELS = '/kaggle/working', ['same_figure','same_paper','related_papers','unrelated_papers']
T0 = time.time()

def find(name):
    """input-এ আগে খোঁজে, না পেলে working-এ — dataset বানানো থাক বা না থাক, চলবে"""
    h = glob.glob(f'/kaggle/input/**/{name}', recursive=True) or glob.glob(f'{OUT}/{name}')
    if not h: raise FileNotFoundError(name)
    return h[0]

W = os.path.dirname(find('emb_clip_img.npy'))
mtr = pd.read_parquet(f'{W}/meta_train.parquet'); mte = pd.read_parquet(f'{W}/meta_test.parquet')
for d in (mtr, mte):
    d['combo'] = np.where(d.t1 < d.t2, d.t1+'+'+d.t2, d.t2+'+'+d.t1)
mtr['y'] = mtr.label.astype(str)

def center(x):
    z = x - x.mean(0, keepdims=True)
    return z / np.linalg.norm(z, axis=1, keepdims=True).clip(1e-8)

ii = {h:i for i,h in enumerate(pd.read_parquet(f'{W}/img_hashes.parquet').hash.values)}
ti = {h:i for i,h in enumerate(pd.read_parquet(f'{W}/txt_hashes.parquet').hash.values)}
SP = {'clip': (np.load(f'{W}/emb_clip_img.npy'), np.load(f'{W}/emb_clip_txt.npy'), ii, ti),
      'sci' : (None, center(np.load(f'{W}/emb_sci_txt.npy')), None, ti),
      'dino': (center(np.load(f'{W}/emb_dino_img.npy')), None, ii, None)}
try:
    S = os.path.dirname(find('emb_siglip_img.npy'))
    SP['sig'] = (np.load(f'{S}/emb_siglip_img.npy'), np.load(f'{S}/emb_siglip_txt.npy'),
        {h:i for i,h in enumerate(pd.read_parquet(f'{S}/img_hashes_siglip.parquet').hash.values)},
        {h:i for i,h in enumerate(pd.read_parquet(f'{S}/txt_hashes_siglip.parquet').hash.values)})
    print('SigLIP:', S)
except FileNotFoundError:
    print('⚠️ SigLIP পাওয়া গেল না — CLIP/SciNCL/DINO দিয়েই চলবে')

HAS = 'sig' in SP
SPACES = {'image+text':  ['clip'] + (['sig'] if HAS else []),
          'text+text':   ['clip','sci'] + (['sig'] if HAS else []),
          'image+image': ['clip','dino'] + (['sig'] if HAS else [])}
print('spaces:', SPACES)

def side(hs, ts, sp):
    ie, te_, mi, mt = SP[sp]
    dim = (ie if ie is not None else te_).shape[1]
    out = np.zeros((len(hs), dim), dtype=np.float32)
    for i, (h, t) in enumerate(zip(hs, ts)):
        if t == 'image' and ie is not None: out[i] = ie[mi[h]]
        elif t == 'text' and te_ is not None: out[i] = te_[mt[h]]
    return out

rng = np.random.default_rng(0)
def build(m, combo):
    d = m[m.combo == combo].reset_index(drop=True)
    B = []
    for sp in SPACES[combo]:
        e1, e2 = side(d.h1.values, d.t1.values, sp), side(d.h2.values, d.t2.values, sp)
        cos = (e1*e2).sum(1)
        cs = []
        for q, pool in ((e1, e2), (e2, e1)):        # hubness: cosine absolute, সীমানা relative
            R = pool[rng.choice(len(pool), min(1500, len(pool)), replace=False)]
            Sm = q @ R.T
            cs += [(cos - Sm.mean(1))/Sm.std(1).clip(1e-6), (Sm < cos[:,None]).mean(1)]
        z1, r1, z2, r2 = cs
        B += [np.abs(e1-e2), e1*e2,
              np.c_[cos, np.linalg.norm(e1-e2, axis=1),
                    np.minimum(z1,z2), np.maximum(z1,z2),
                    np.minimum(r1,r2), np.maximum(r1,r2)].astype(np.float32)]
    mn = np.minimum(d.len1, d.len2).values; mx = np.maximum(d.len1, d.len2).values
    B.append(np.c_[mn, mx, mx/np.maximum(mn,1)].astype(np.float32))
    return d, np.hstack(B).astype(np.float32)

def coord_ascent(P, ytrue, classes, rounds=10):
    """argmax macro-F1-এর জন্য optimal নয় — per-class multiplier greedy খুঁজি"""
    w = np.ones(len(classes))
    best = f1_score(ytrue, [classes[i] for i in (P*w).argmax(1)], average='macro')
    for _ in range(rounds):
        moved = False
        for j in range(len(classes)):
            for m in (0.7,0.85,0.95,1.05,1.15,1.35):
                w2 = w.copy(); w2[j] *= m
                s = f1_score(ytrue, [classes[i] for i in (P*w2).argmax(1)], average='macro')
                if s > best + 1e-5: best, w, moved = s, w2, True
        if not moved: break
    return w, best

sub = pd.DataFrame({'id': mte.id.values})
for c in LABELS: sub[c] = 0
report, sizes = {}, {}

for combo in ['image+text','text+text','image+image']:
    d,  X  = build(mtr, combo)
    dt, Xt = build(mte, combo)
    classes = sorted(d.y.unique()); K = len(classes)
    y = d.y.map({c:i for i,c in enumerate(classes)}).values
    oof = np.zeros((len(y),K)); test = np.zeros((len(dt),K))

    for f,(tr_i,va_i) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(X,y)):
        g = lgb.LGBMClassifier(objective='multiclass', num_class=K, n_estimators=700,
                               learning_rate=0.05, num_leaves=31, colsample_bytree=0.3,
                               subsample=0.8, subsample_freq=1, class_weight='balanced',
                               verbose=-1, n_jobs=-1).fit(np.vstack([X[tr_i]]*2),
                                                          np.concatenate([y[tr_i]]*2))
        oof[va_i] = g.predict_proba(X[va_i]); test += g.predict_proba(Xt)/5
        print(f'  {combo} fold {f+1}/5  {time.time()-T0:.0f}s', flush=True)

    raw = f1_score(d.y, [classes[i] for i in oof.argmax(1)], average='macro')
    mult, thr = coord_ascent(oof, d.y.values, classes)
    print(f'\n### {combo}  argmax {raw:.4f} → +threshold {thr:.4f}')
    print(classification_report(d.y, [classes[i] for i in (oof*mult).argmax(1)], digits=3))
    report[combo] = {'spaces': SPACES[combo], 'argmax': round(raw,4), 'final': round(thr,4)}
    sizes[combo] = len(d)

    pos = pd.Index(sub.id).get_indexer(dt.id.values)
    assert (sub.loc[pos,'id'].values == dt.id.values).all()
    for i, j in enumerate((test*mult).argmax(1)):
        sub.loc[pos[i], classes[j]] = 1
    # blend-এর কাঁচামাল — ছোট .npy, pickle নয় (প্রতিটা ~150 KB)
    np.save(f'{OUT}/B_oof_{combo.replace("+","_")}.npy',  oof.astype(np.float32))
    np.save(f'{OUT}/B_test_{combo.replace("+","_")}.npy', test.astype(np.float32))

sub[['id']+LABELS].to_csv(f'{OUT}/submission.csv', index=False)
tot = sum(sizes.values())
report['overall'] = round(sum(report[c]['final']*sizes[c] for c in sizes)/tot, 4)
report['baseline_planA'] = 0.4806
json.dump(report, open(f'{OUT}/report_planB.json','w'), indent=1)
print('\n' + '='*60); print(json.dumps(report, indent=1))
print('\nsubmission:', sub[LABELS].sum().to_dict())
print('প্রতি row-তে একটা 1:', sub[LABELS].sum(1).value_counts().to_dict())
print(f'মোট {(time.time()-T0)/60:.1f} মিনিট  →  {OUT}/submission.csv')
